# NYC Taxi Fare Prediction - Phase 5: Model Building
### Automatidata x New York City Taxi & Limousine Commission
---
**Goal:** Train and compare three regression models, Linear Regression, Random Forest, and XGBoost, to predict `fare_amount` from the engineered feature matrix.

**Operations (in order):**
1. Load the engineered feature matrix
2. Split into train/test sets (80/20)
3. Wrap each model in a log1p target transform so evaluation happens in dollar scale
4. Fit Linear Regression -> Random Forest -> XGBoost
5. Compare all three on RMSE, MAE, R²

**Input:**  `data/taxi_features.parquet` — 991,998 rows × 19 columns  
**Output:** Three fitted models + a comparison table, carried into Phase 6 for deeper evaluation

In [ ]:
# Step 5.1 — Notebook Initialization

import sys
sys.path.append("..")

from src.config import *

from sklearn.compose import TransformedTargetRegressor

# Load engineered feature matrix
FEATURES_PATH: Path = DATA_DIR / "taxi_features.parquet"

df = pd.read_parquet(FEATURES_PATH)

# Confirm baseline
print("Feature Matrix Baseline")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")
print(f"\n  Columns: {df.columns.tolist()}")

print(f"\nTarget Variable Check")
print(f"  Target column: {TARGET_COL}")
print(f"  Target present: {TARGET_COL in df.columns}")
print(f"  Target dtype: {df[TARGET_COL].dtype}")
print(f"  Target min/max: ${df[TARGET_COL].min():.2f} / ${df[TARGET_COL].max():.2f}")

print(f"\nFeature Dtypes")
print(df.dtypes.to_string())

## Step 5.2: Train/Test Split
Split the feature matrix into training (80%) and test (20%) sets. The test set is held out completely until Phase 6, no preprocessing statistics (scaling, encoding) are ever fit on it. `fare_amount` is separated into
`y` and kept in its raw dollar scale; the log1p transform happens inside each model's pipeline, not here.

In [ ]:
# ------------------------------------------------------------
# Step 5.2: Train/Test Split
# ------------------------------------------------------------

def split_features_target(
    df: pd.DataFrame,
    target_col: str = TARGET_COL,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Separate the feature matrix into predictors (X) and target (y).

    Parameters
    ----------
    df: pd.DataFrame
        The full engineered dataset.
    target_col: str
        Name of the target column to separate out.

    Returns
    -------
    tuple[pd.DataFrame, pd.Series]
        X (all columns except target) and y (target column), in
        raw dollar scale — no transformation applied here.
    """
    X = df.drop(columns=[target_col])
    y = df[target_col].copy()

    return X, y

# Run split
X, y = split_features_target(df, target_col=TARGET_COL)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = TEST_SIZE,
    random_state = RANDOM_STATE
)

logger.info(f"Train set: {X_train.shape[0]:,} rows x {X_train.shape[1]} cols")
logger.info(f"Test set: {X_test.shape[0]:,} rows x {X_test.shape[1]} cols")

# Verify
print("\n" + "=" * 75)
print("Split Summary")
print("=" * 75)

print(f"  Total rows: {len(df):,}")
print(f"  Train rows: {len(X_train):,} ({len(X_train)/len(df)*100:.1f}%)")
print(f"  Test rows: {len(X_test):,} ({len(X_test)/len(df)*100:.1f}%)")
print(f"  Features: {X_train.shape[1]}")

print("\n" + "=" * 75)
print("Target Distribution Check (train vs test)")
print("=" * 75)

print(f"  {'Metric':<10} {'Train':>12} {'Test':>12}")
print("-" * 40)

for stat_name, fn in [
    ("Mean", np.mean),
    ("Median", np.median),
    ("Std", np.std),
    ("Min", np.min),
    ("Max", np.max)
]:
    print(f"  {stat_name:<10} {fn(y_train):>12.2f} {fn(y_test):>12.2f}")

print("\n" + "=" * 75)
print("Feature Columns")
print("=" * 75)

print(X_train.columns.tolist())


In [ ]:
print(TARGET_COL)

print(y_train.name)
print(y_train.head(10))
print(y_train.dtype)

## Step 5.2: Train/Test Split ✅

Feature matrix split 80/20 into train and test sets, stratification not needed since the target is continuous and the split is large enough to preserve the distribution naturally.

### Split Summary

| Set | Rows | % |
|---|---|---|
| Train | 793,598 | 80.0% |
| Test | 198,400 | 20.0% |
| Total | 991,998 | 100% |

### Target Distribution — Train vs Test

| Metric | Train | Test |
|---|---|---|
| Mean | \$13.11 | \$13.03 |
| Median | $9.50 | \$9.50 |
| Std | \$11.49 | \$11.30 |
| Min | \$0.01 | \$0.01 |
| Max | \$621.50 | \$325.50 |

**Observations:**
- Mean, median, and std are all closely matched between train and test, the split is representative, no re-shuffling or stratification needed
- The max fare differs ($621.50 train vs $325.50 test) simply because the handful of extreme outlier trips (>$300) happened to land in the train
  set by chance, with only a few hundred such trips in a 991,998-row dataset, this is expected sampling variance and not a concern
- 18 features confirmed, `fare_amount` correctly isolated as `y`

### Feature Columns (18)

`extra`, `pickup_hour`, `pickup_day_of_week`, `is_weekend`, `is_rush_hour`,
`is_overnight`, `log1p_trip_distance`, `log1p_tolls_amount`,
`log1p_trip_duration_min`, `rate_group_ride`, `rate_jfk`, `rate_nassau_wc`,
`rate_negotiated`, `rate_newark`, `vendor_verifone`, `pax_large_group`,
`pax_small_group`, `is_credit_card`

## Step 5.3: Preprocessing Pipeline
Build the shared preprocessing components used across all three models: a `TransformedTargetRegressor` wrapper that log1p-transforms `fare_amount` before fitting and automatically applies `expm1` to inverse-transform predictions back to dollar scale. A `StandardScaler` step is added only for the Linear Regression pipeline, tree-based models (Random Forest, XGBoost) are scale-invariant and don't need it.

In [ ]:
# Step 5.3: Preprocessing Pipeline

def build_linear_pipeline(random_state: int = RANDOM_STATE) -> TransformedTargetRegressor:
    """
    Build a scaled Linear Regression pipelines wrapped in a log1p target transform.

    The target is log1p-transformed before fitting and expm1-transformed on prediction,
    so ``.predict()`` always returns values in a raw dollar scale. Features are standard-scaled
    since Linear Regression is sensitive to feature magnitude.

    Parameters
    ----------
    random_state: int
        Unused for Linear Regression (no randomness), kept for a
        consistent function signature across all three model builders.

    Returns
    -------
    TransformedTargetRegressor
        Unfitted pipeline: StandardScaler -> LinearRegression, with
        log1p/expm1 target transform applied automatically.
    """

    base_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    return TransformedTargetRegressor(
        regressor = base_pipeline,
        func = np.log1p,
        inverse_func = np.expm1
    )

def build_random_forest_pipeline(
    random_state: int = RANDOM_STATE,
    n_estimators: int = 100,
    max_depth: Optional[int] = None,
    n_jobs: int = -1
) -> TransformedTargetRegressor:
    """
    Build a Random Forest pipeline wrapped in a log1p target transform.

    No feature scaling is applied as tree-based splits are invariant to
    monotonic transformations of feature scale.

    Parameters
    ----------
    random_state: int
        Seed for reproducibility.
    n_estimators: int
        Number of trees in the forest (default: 100).
    max_depth: Optional[int]
        Maximum tree depth. none allows full growth (default: None).
    n_jobs: int
        Parallel jobs for fitting (default: -1, uses all cores).

    Returns
    -------
    TransformedTargetRegressor
        Unfitted RandomForestRegressor with log1p/expm1 target transform.
    """
    model = RandomForestRegressor(
        n_estimators = n_estimators,
        max_depth = max_depth,
        random_state = random_state,
        n_jobs = n_jobs
    )

    return TransformedTargetRegressor(
        regressor = model,
        func = np.log1p,
        inverse_func = np.expm1
    )

def build_xgboost_pipeline(
    random_state: int = RANDOM_STATE,
    n_estimators: int = 300,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    n_jobs: int = -1
) -> TransformedTargetRegressor:
    """
    Build an XGBoost pipeline wrapped in a log1p target transform.

    Parameters
    ----------
    random_state: int
        Seed for reproducibility.
    n_estimators: int
        Number of boosting rounds (default 300).
    max_depth: int
        Maximum tree depth (default 6).
    learning_rate: float
        Step size shrinkage (default 0.1).
    n_jobs: int
        Parallel jobs for fitting (default: -1, uses all cores).

    Returns
    -------
    TransformedTargetRegressor
        Unfitted XGBRegressor with log1p/expm1 target transform.
    """
    model = xgb.XGBRegressor(
        n_estimators = n_estimators,
        max_depth = max_depth,
        learning_rate = learning_rate,
        random_state = random_state,
        n_jobs = n_jobs
    )

    return TransformedTargetRegressor(
        regressor = model,
        func = np.log1p,
        inverse_func = np.expm1
    )

# Build all three (unfitted)
linear_pipeline = build_linear_pipeline()
rf_pipeline = build_random_forest_pipeline()
xgb_pipeline = build_xgboost_pipeline()

print("\n" + "=" * 50)
print("Pipelines Built")
print("=" * 50)

print(f"  Linear Regression: {linear_pipeline}")
print(f"  Random Forest: {rf_pipeline}")
print(f"  XGBoost: {xgb_pipeline}")

print("\n" + "=" * 50)
print("Target Transform Sanity Check")
print("=" * 50)

sample = y_train.iloc[:5].values
transformed = np.log1p(sample)
inverted = np.expm1(transformed)

print(f"  Original: {sample}")
print(f"  log1p: {np.round(transformed, 4)}")
print(f"  expm1 back: {np.round(inverted, 4)}")
print(f"  Roundtrip match: {np.allclose(sample, inverted)}")


## Step 5.3: Preprocessing Pipeline ✅

Three unfitted pipelines built, each wrapped in a `TransformedTargetRegressor` so `fare_amount` is log1p-transformed before fitting and automatically inverse-transformed (`expm1`) on prediction — every model returns dollar-scale predictions with no manual conversion needed downstream.

### Pipeline Configuration

| Model | Preprocessing | Estimator | Key Params |
|---|---|---|---|
| Linear Regression | `StandardScaler` | `LinearRegression` | defaults |
| Random Forest | None (scale-invariant) | `RandomForestRegressor` | `n_estimators=100`, `random_state=42` |
| XGBoost | None (scale-invariant) | `XGBRegressor` | `n_estimators=300`, `max_depth=6`, `learning_rate=0.1` |

### Target Transform Sanity Check

| Original ($) | log1p | expm1 (back) |
|---|---|---|
| 21.50 | 3.1135 | 21.50 |
| 12.50 | 2.6027 | 12.50 |
| 6.50 | 2.0149 | 6.50 |
| 8.50 | 2.2513 | 8.50 |
| 10.00 | 2.3979 | 10.00 |

**Roundtrip match: ✅ True**: confirms the transform is lossless and every RMSE/MAE/R² computed in Phase 6 will be in real dollars, not log-dollars.

**Observations:**
- Only Linear Regression gets `StandardScaler`. Random Forest and XGBoost split on raw feature values and are unaffected by scale
- Random Forest uses 100 trees as a sensible default baseline, XGBoost uses 300 rounds at a conservative `learning_rate=0.1`. Both can be tuned further in Phase 6 if needed
- `TransformedTargetRegressor` means `.fit(X, y)` and `.predict(X)` handle the log1p/expm1 conversion internally, no changes needed to the standard sklearn API.

## Step 5.4: Baseline (Linear Regression)
Fit the Linear Regression pipeline on the training set and evaluate on the test set. This establishes the baseline that Random Forest and XGBoost must beat to justify their added complexity.

In [ ]:
# Step 5.4: Baseline (Linear Regression)

def evaluate_model(
    model: TransformedTargetRegressor,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    model_name: str
) -> dict:
    """
    Evaluate a fitted model on the test set using RMSE, MAE and R².

    Predictions are already in dollar scale since the models ``TransformedTargetRegressor``
    wrapper applies the inverse transform automatically.

    Parameters
    ----------
    model: TransformedTargetRegressor
        A fitted model pipeline.
    X_test: pd.DataFrame
        Held-out test features.
    y_test: pd.Series
        Held-out test target, in raw dollar scale.
    model_name: str
        Label used for logging and the returned results dict.

    Returns
    -------
    dict
        Model name, RMSE, MAE, and R² on the test set.
    """
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    logger.info(f"{model_name} | RMSE: ${rmse:.3f} | MAE: ${mae:.3f} | R²: {r2:.4f}")

    return {
        "model": model_name,
        "rmse": round(rmse, 4),
        "mae": round(mae, 4),
        "r2": round(r2, 4)
    }

# Fit
logger.info("Fitting Linear Regression...")
linear_pipeline.fit(X_train, y_train)
logger.info("Linear Regression fit complete")

# Predict & Evaluate
linear_results = evaluate_model(linear_pipeline, X_test, y_test, "Linear Regression")

print("\n" + "=" * 50)
print("Linear Regression Results")
print("=" * 50)

print(f"  RMSE: ${linear_results['rmse']:.3f}")
print(f"  MAE: ${linear_results['mae']:.3f}")
print(f"  R²: {linear_results['r2']:.4f}")

# Coefficient inspection (in log1p-target space)
coefs = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": linear_pipeline.regressor_.named_steps["model"].coef_
}).sort_values("coefficient", key=abs, ascending=False)

print("\n" + "=" * 50)
print("Top 10 Coefficients (log1p-fare space)")
print("=" * 50)

print(coefs.head(10).to_string(index=False))

print("\n" + "=" * 50)
print("Sample Predictions vs Actual")
print("=" * 50)

sample_idx = X_test.index[:8]
sample_preds = pd.DataFrame({
    "actual": y_test.loc[sample_idx].values,
    "predicted": linear_pipeline.predict(X_test.loc[sample_idx])
})
sample_preds["error"] = (sample_preds["predicted"] - sample_preds["actual"]).round(2)
print(sample_preds.round(2).to_string(index=False))


## Step 5.4: Baseline (Linear Regression) ✅

### Test Set Performance

| Metric | Value |
|---|---|
| RMSE | \$3.848 |
| MAE | \$0.913 |
| R² | 0.8841 |

### Top 10 Coefficients (standardized, log1p-fare space)

| Feature | Coefficient |
|---|---|
| `log1p_trip_distance` | 0.3476 |
| `log1p_trip_duration_min` | 0.2377 |
| `rate_negotiated` | 0.0393 |
| `rate_jfk` | 0.0222 |
| `rate_newark` | 0.0205 |
| `is_weekend` | -0.0107 |
| `log1p_tolls_amount` | 0.0100 |
| `rate_nassau_wc` | 0.0096 |
| `pickup_day_of_week` | 0.0080 |
| `extra` | -0.0048 |

### Sample Predictions vs Actual

| Actual | Predicted | Error |
|---|---|---|
| \$11.00 | \$11.28 | +\$0.28 |
| \$4.50 | \$4.79 | +\$0.29 |
| \$5.00 | \$5.09 | +\$0.09 |
| \$8.50 | \$8.83 | +\$0.33 |
| \$4.00 | \$4.19 | +\$0.19 |
| \$7.50 | \$7.38 | -\$0.12 |
| \$14.50 | \$14.13 | -\$0.37 |
| \$8.00 | \$8.38 | +\$0.38 |

**Observations:**
- R² of 0.884 means the linear model explains 88.4\% of fare variance, a strong baseline given only 18 features and no interaction terms
- MAE of \\$0.91 is small relative to the median fare of \$9.50, the model is off by less than 10% on a typical trip
- Because features were standard-scaled, coefficients are directly comparable: `log1p_trip_distance` (0.348) and `log1p_trip_duration_min` (0.238) dominate, together accounting for most of the model's predictive power, consistent with their Pearson correlations (0.887 and 0.743) from Phase 4
- `rate_jfk`'s coefficient (0.022) looks modest next to its raw Pearson correlation (0.542) from Phase 4. This is expected, not a contradiction. Once `log1p_trip_distance` and `log1p_trip_duration_min` are in the model, they already explain most of why JFK trips cost more (they're long trips). The coefficient captures JFK's *marginal* effect on top of distance and duration, not its total association with fare.
- `rate_negotiated` carries the largest categorical coefficient (0.039), consistent with Phase 2's finding that negotiated fares have the widest variance and can command high flat rates
- RMSE (\$3.85) being over 4x MAE (\\$0.91) signals a right-skewed error distribution, a small number of large errors (likely long or irregular trips) are pulling RMSE up while most predictions are tight, consistent with squared-error's sensitivity to outliers
- Sample predictions show errors mostly under \$0.40 in either direction, tight and unbiased at a glance, no systematic over- or under-prediction visible in this small sample.

## Step 5.5: Random Forest
Fit the Random Forest pipeline and compare against the Linear Regression baseline. As an ensemble of decision trees, Random Forest can capture non-linear relationships and feature interactions (e.g. rush hour × trip distance) that a linear model cannot.

In [ ]:
# Step 5.5: Random Forest

# Fit
logger.info("Fitting Random Forest...")
rf_pipeline.fit(X_train, y_train)
logger.info("Random Forest fit complete.")

# Predict & Evaluate
rf_results = evaluate_model(rf_pipeline, X_test, y_test, "Random Forest")

print("\n" + "=" * 50)
print("Random Forest Results")
print("=" * 50)

print(f"  RMSE: ${rf_results['rmse']:.3f}")
print(f"  MAE: ${rf_results['mae']:.3f}")
print(f"  R²: {rf_results['r2']:.4f}")

# Feature importance
rf_importances = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_pipeline.regressor_.feature_importances_
}).sort_values("importance", ascending=False)

print("\n" + "=" * 50)
print("Top 10 Feature Importances")
print("=" * 50)

print(rf_importances.head(10).to_string(index=False))

# Comparison vs Linear Regression
print("\n" + "=" * 50)
print("Random Forest vs Linear Regression")
print("=" * 50)

print(f"  {'Metric':<8} {'Linear':>10} {'Random Forest':>15} {'Change':>10} {'Better?':>8}")
print(f"  {'-' * 55}")
for metric in ["rmse", "mae", "r2"]:
    lin_val = linear_results[metric]
    rf_val = rf_results[metric]
    delta = rf_val - lin_val
    pct = (delta / lin_val * 100)
    if metric == "r2":
        improved = delta > 0
    else:
        improved = delta < 0
    print(f"  {metric.upper():<8} {lin_val:>10.4f} {rf_val:>15.4f} {pct:>+9.2f}% {'✅' if improved else '❌':>8}")

print("\n" + "=" * 50)
print("Sample Predictions vs Actual")
print("=" * 50)

sample_preds_rf = pd.DataFrame({
    "actual": y_test.loc[sample_idx].values,
    "predicted": rf_pipeline.predict(X_test.loc[sample_idx])
})
sample_preds_rf["error"] = (sample_preds_rf["predicted"] - sample_preds_rf["actual"]).round(2)
print(sample_preds_rf.round(2).to_string(index=False))


## Step 5.5: Random Forest ✅

### Test Set Performance

| Metric | Value |
|---|---|
| RMSE | \\$2.154 |
| MAE | \\$0.343 |
| R² | 0.9637 |

### Top 10 Feature Importances

| Feature | Importance |
|---|---|
| `log1p_trip_distance` | 0.7648 |
| `log1p_trip_duration_min` | 0.2140 |
| `rate_jfk` | 0.0098 |
| `rate_negotiated` | 0.0059 |
| `pickup_hour` | 0.0012 |
| `log1p_tolls_amount` | 0.0011 |
| `is_credit_card` | 0.0008 |
| `rate_newark` | 0.0006 |
| `pickup_day_of_week` | 0.0005 |
| `vendor_verifone` | 0.0003 |

### Random Forest vs Linear Regression

| Metric | Linear | Random Forest | Change | Better? |
|---|---|---|---|---|
| RMSE | \\$3.848 | \\$2.154 | -44.02\% | ✅ |
| MAE | \\$0.914 | \\$0.343 | -62.45\% | ✅ |
| R² | 0.8841 | 0.9637 | +9.00\% | ✅ |

### Sample Predictions vs Actual

| Actual | Predicted | Error |
|---|---|---|
| \\$11.00 | \\$11.14 | +\\$0.14 |
| \\$4.50 | \\$4.55 | +\\$0.05 |
| \\$5.00 | \\$4.99 | -\\$0.01 |
| \\$8.50 | \\$8.67 | +\\$0.17 |
| \\$4.00 | \\$4.33 | +\\$0.33 |
| \\$7.50 | \\$7.14 | -\\$0.36 |
| \\$14.50 | \\$13.76 | -\\$0.74 |
| \\$8.00 | \\$8.25 | +\\$0.25 |

**Observations:**
- Random Forest beats Linear Regression decisively on every metric, a 44\% RMSE reduction and 62\% MAE reduction is a large jump for switching model families on the same 18 features
- `log1p_trip_distance` (0.765) and `log1p_trip_duration_min` (0.214) together account for **97.9\%** of total importance. The model is leaning almost entirely on two features, with everything else contributing marginally
- This importance concentration mirrors the Linear Regression coefficients, but Random Forest extracts far more signal from the same two features by modeling non-linear thresholds and interactions (e.g. how the distance-fare relationship bends differently for short vs long trips) that a single linear coefficient can't represent
- `rate_jfk` importance (0.0098) looks small in isolation, but that's consistent with Step 5.4's finding: once trip distance and duration are available, the model already captures most of what makes JFK trips expensive (they're long trips). The flat rate's marginal contribution is real but modest on top of that
- Comparing the same 8 sample rows against Linear Regression: most errors shrank (e.g. \\$7.50 actual: -\\$0.12 -> -\\$0.36 is actually worse here, but \\$11.00 actual: +\\$0.28 -> +\\$0.14 improved). MAE improving overall doesn't mean every single prediction gets better, just that the typical error shrinks in aggregate
- MAE of \\$0.34 against a median fare of \\$9.50 means the model is typically within ~3.6\\% of the true fare, strong performance for a pre-tuning baseline configuration.

## Step 5.6: XGBoost
Fit the XGBoost pipeline and compare against both prior models. As a gradient-boosted ensemble, XGBoost builds trees sequentially, each one correcting the errors of the previous, often edging out Random Forest on tabular data, though not guaranteed given Random Forest's already strong performance here.

In [ ]:
# Step 5.6: XGBoost

# Fit
logger.info("Fitting XGBoost...")
xgb_pipeline.fit(X_train, y_train)
logger.info("XGBoost fit complete.")

# Predict & Evaluate
xgb_results = evaluate_model(xgb_pipeline, X_test, y_test, "XGBoost")

print("\n" + "=" * 50)
print("XGBoost Resutls")
print("=" * 50)
print(f"  RMSE: ${xgb_results['rmse']:.3f}")
print(f"  MAE: ${xgb_results['mae']:.3f}")
print(f"  R²: {xgb_results['r2']:.4f}")

# Feature importance (gain-based)
xgb_importances = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_pipeline.regressor_.feature_importances_
}).sort_values("importance", ascending=False)

print("\n" + "=" * 50)
print("Top 10 Feature Importances (gain)")
print("=" * 50)
print(xgb_importances.head(10).to_string(index=False))

# Comparison vs both prior models
print("\n" + "=" * 50)
print("XGBoost vs Random Forest vs Linear Regression")
print("=" * 50)
print(f"  {'Metric':<8} {'Linear Regression':>15} {'Random Forest':>15} {'XGBoost':>10} {'RF->XGB Δ':>10}")
print(f"  {'-' * 60}")
for metric in ["rmse", "mae", "r2"]:
    lin_val = linear_results[metric]
    rf_val = rf_results[metric]
    xgb_val = xgb_results[metric]
    delta = xgb_val - rf_val
    pct = delta / rf_val * 100
    print(f"  {metric.upper():<8} {lin_val:>15.4f} {rf_val:>15.4f} {xgb_val:>10.4f} {pct:>+9.2f}%")

print("\n" + "=" * 50)
print("Sample Predictions vs Actual")
print("=" * 50)
sample_preds_xgb = pd.DataFrame({
    "actual": y_test.loc[sample_idx].values,
    "predicted": xgb_pipeline.predict(X_test.loc[sample_idx])
})
sample_preds_xgb["error"] = (sample_preds_xgb["predicted"] - sample_preds_xgb["actual"]).round(2)
print(sample_preds_xgb.round(2).to_string(index=False))


## Step 5.6: XGBoost ✅

### Test Set Performance

| Metric | Value |
|---|---|
| RMSE | \$2.076 |
| MAE | \$0.354 |
| R² | 0.9663 |

### Top 10 Feature Importances (gain-based)

| Feature | Importance |
|---|---|
| `log1p_trip_distance` | 0.6756 |
| `log1p_trip_duration_min` | 0.2072 |
| `rate_jfk` | 0.0602 |
| `rate_negotiated` | 0.0155 |
| `rate_newark` | 0.0133 |
| `vendor_verifone` | 0.0069 |
| `is_credit_card` | 0.0067 |
| `rate_nassau_wc` | 0.0035 |
| `log1p_tolls_amount` | 0.0034 |
| `is_rush_hour` | 0.0022 |

### Three-Way Model Comparison

| Metric | Linear Regression | Random Forest | XGBoost | RF->XGB Change |
|---|---|---|---|---|
| RMSE | \$3.848 | \$2.154 | \$2.076 | -3.62% ✅ |
| MAE | \$0.914 | \$0.343 | \$0.354 | +3.12% ❌ |
| R² | 0.8841 | 0.9637 | 0.9663 | +0.27% ✅ |

**Observations:**
- XGBoost improves on Random Forest for RMSE and R², but **MAE is slightly worse** (\$0.354 vs \$0.343). This isn't a contradiction. RMSE penalizes large errors quadratically, so XGBoost is doing a better job on the tail (large-fare, harder-to-predict trips), at the cost of being marginally less accurate on the typical small-fare trip that MAE weights equally. The sample predictions below show this: the \\$14.50 trip has a bigger XGBoost error (-\\$0.85 vs -\\$0.74 for Random Forest)
- `rate_jfk` importance jumps from 0.0098 (Random Forest) to 0.0602 in XGBoost, gradient boosting's sequential correction process appears to extract more marginal signal from the JFK flat-rate flag than Random Forest's independent trees do
- `log1p_trip_distance` + `log1p_trip_duration_min` still dominate at 88.3% combined importance, slightly less concentrated than Random Forest's 97.9%. XGBoost distributes more weight across secondary features (`rate_jfk`, `rate_negotiated`, `rate_newark`, `vendor_verifone`)
- Fit time was roughly 5 seconds vs Random Forest's ~28 seconds. XGBoost's shallower trees (`max_depth=6`) and boosting structure are considerably cheaper here despite 3x more estimators (300 vs 100)
- Given XGBoost wins 2 of 3 metrics by a small margin and Random Forest wins MAE by a small margin, the practical difference between the two is minor at these default hyperparameters. A genuine choice would depend on whether the business cares more about typical-case accuracy (MAE -> Random Forest) or capping worst-case error (RMSE -> XGBoost).

## Step 5.7: Model Comparison
Consolidate all three models into a single comparison table and visualize performance side by side. This closes out Phase 5 with a clear picture of which model to carry forward for deeper evaluation in Phase 6.

In [ ]:
# Step 5.7: Model Comparison

def build_comparison_table(results_list: list[dict]) -> pd.DataFrame:
    """
    Consolidate multiple model result dicts into a single comparison table.

    Parameters
    ----------
    results_list: list[dict]
        List of result dicts, each produced by ``evaluate_model()``
        with keys ``model``, ``rmse``, ``mae``, ``r2``.

    Returns
    -------
    pd.DataFrame
        Comparison table sorted by RMSE ascending (best model first),
        with a rank column added.
    """
    comparison = pd.DataFrame(results_list)
    comparison = comparison.sort_values("rmse", ascending=True).reset_index(drop=True)
    comparison.insert(0, "rank", range(1, len(comparison) + 1))

    return comparison

def plot_model_comparison(comparison: pd.DataFrame) -> None:
    """
    Plot a 3-panel bar chart comparing RMSE, MAE and R² across models.

    Parameters
    ----------
    comparison: pd.DataFrame
        Output of ``build_comparison_table()``.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Model Comparison: Test Set Performance", fontsize=15)

    metrics = [
        ("rmse", "RMSE ($): lower is better", PALETTE["accent"]),
        ("mae",  "MAE ($): lower is better", PALETTE["primary"]),
        ("r2",   "R²: higher is better", PALETTE["secondary"]),
    ]

    for ax, (metric, title, color) in zip(axes, metrics):
        bars = ax.bar(
            comparison["model"],
            comparison[metric],
            color=color, edgecolor="white", linewidth=0.4
        )
        for bar, val in zip(bars, comparison[metric]):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{val:.3f}" if metric != "r2" else f"{val:.4f}",
                ha="center", va="bottom", fontsize=10
            )
        ax.set_title(title)
        ax.set_ylabel(metric.upper())
        ax.tick_params(axis="x", rotation=15)

    plt.tight_layout()
    save_figure(fig, "phase5_model_comparison")
    plt.show()

# Build comparison
all_results = [linear_results, rf_results, xgb_results]
comparison_table = build_comparison_table(all_results)

print("\n" + "=" * 50)
print("Final Model Comparison")
print("=" * 50)
print(comparison_table.to_string(index=False))

print("\n" + "=" * 50)
print("Best Model by Metric")
print("=" * 50)
print(f"  Lowest RMSE: {comparison_table.loc[comparison_table['rmse'].idxmin(), 'model']}")
print(f"  Lowest MAE: {comparison_table.loc[comparison_table['mae'].idxmin(), 'model']}")
print(f"  Highest R²: {comparison_table.loc[comparison_table['r2'].idxmax(), 'model']}")

plot_model_comparison(comparison_table)


## Step 5.7: Model Comparison ✅

### Final Comparison Table

| Rank | Model | RMSE | MAE | R² |
|---|---|---|---|---|
| 1 | XGBoost | \$2.076 | \$0.354 | 0.9663 |
| 2 | Random Forest | \$2.154 | \$0.343 | 0.9637 |
| 3 | Linear Regression | \$3.848 | \$0.914 | 0.8841 |

### Best Model by Metric

| Metric | Winner |
|---|---|
| Lowest RMSE | XGBoost |
| Lowest MAE | Random Forest |
| Highest R² | XGBoost |

### Plot Observations

All three panels tell a consistent story: Linear Regression is a clear step behind both ensemble methods on every metric, while XGBoost and Random Forest sit close together with a narrow, metric-dependent gap.

- **RMSE panel:** Linear Regression's bar (\\$3.848) towers nearly 2x above both ensembles, visually confirming it can't capture the non-linear distance/duration interactions and rate-code step-changes that drive fare variance. XGBoost (\\$2.076) and Random Forest (\\$2.154) are close, with XGBoost narrowly ahead
- **MAE panel:** Same large gap for Linear Regression (\\$0.914), but here Random Forest (\\$0.343) edges out XGBoost (\\$0.354). The only panel where the ranking flips
- **R² panel:** All three bars sit close to 1.0 on an absolute scale, which can visually understate the practical gap, the table is the better read here, 0.884 vs 0.964-0.966 is a meaningful difference in explained variance

**Observations:**
- No single model wins outright. XGBoost takes RMSE and R², Random Forest takes MAE, and the margins are small (RMSE differs by \\$0.078, MAE by \\$0.011)
- Linear Regression is unambiguously the weakest of the three across every metric and every panel. It's a useful interpretable baseline but not viable as the production model given this data
- Given the RMSE/MAE split doesn't favor one model conclusively, Phase 6's residual analysis (where errors are large, whether they cluster by trip type or fare range) is the right next step before making a final recommendation, rather than picking a winner on these three aggregate numbers alone